# Práctica 4 · Mejorar lo que llega al modelo

En las prácticas anteriores quedaron dos piezas marcadas como pendientes en las figuras: la
búsqueda híbrida y el reordenamiento de resultados. Hoy las construimos.

Las dos atacan el mismo problema, que es el cuello de botella real de un sistema RAG. El modelo
solo puede responder con lo que le llega; si la búsqueda trae lo que no era, no hay prompt ni
modelo que lo arregle. Así que en lugar de seguir puliendo la respuesta, vamos a mejorar la
selección.

Y hay algo más que quiero que te lleves de esta práctica, aparte de las dos técnicas. Vamos a
medir si de verdad sirven en nuestro caso, y una de las dos va a salir mal parada. Eso también
es parte del trabajo: saber cuándo una técnica famosa no te está aportando nada.

Las celdas se ejecutan en orden, una por una, con Shift + Enter.

## 1. Instalar las librerías

Una sola librería nueva: `rank_bm25`, que implementa la búsqueda por palabras. Es minúscula y
solo depende de numpy, a diferencia de las alternativas industriales que exigen un servidor de
búsqueda aparte.

In [1]:
%pip install --quiet rank_bm25 pymupdf

print("Librerías listas.")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/fgrodriguez/ESAN_GlobalWeek2026/09_Notebooks_RAG/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Librerías listas.


## 2. Comprobar que Ollama responde

Ollama es el programa que ejecuta los modelos de lenguaje dentro de tu computadora. Tiene que
estar encendido para que este cuaderno funcione, así que lo primero es confirmarlo.

Si algo falla, la salida de la celda te dice qué hacer según tu sistema operativo.

In [2]:
import platform
import sys

import requests

OLLAMA_URL = "http://localhost:11434"

print(f"Sistema: {platform.system()} {platform.machine()}")
print(f"Python:  {sys.version.split()[0]}\n")

try:
    respuesta = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    respuesta.raise_for_status()
    modelos = sorted(m["name"] for m in respuesta.json()["models"])
    print(f"Ollama responde. Tienes {len(modelos)} modelos descargados:\n")
    for m in modelos:
        print(f"  - {m}")
except Exception as e:
    print(f"Ollama no responde en {OLLAMA_URL}")
    print(f"Detalle: {type(e).__name__}\n")
    if platform.system() == "Darwin":
        print("En Mac: abre la aplicación Ollama desde la carpeta Aplicaciones.")
        print("Debe aparecer su ícono en la barra de menús, arriba a la derecha.")
    elif platform.system() == "Windows":
        print("En Windows: busca Ollama en el menú Inicio y ábrelo.")
        print("Debe aparecer su ícono junto al reloj, abajo a la derecha.")
    else:
        print("Ejecuta 'ollama serve' en una terminal.")

Sistema: Darwin arm64
Python:  3.12.13

Ollama responde. Tienes 14 modelos descargados:

  - embeddinggemma:300m
  - gemma3:1b
  - gemma3:4b
  - gemma4:12b-mlx
  - gemma4:26b
  - gemma4:e4b
  - granite4.1:3b
  - granite4.1:8b
  - mxbai-embed-large:latest
  - nemotron-mini:4b
  - nomic-embed-text:latest
  - qwen3-4b-cs-ft:latest
  - qwen3:4b
  - shieldgemma:2b


## 3. Elegir el modelo según tu equipo

El modelo va a correr en tu máquina, así que la memoria que tengas importa. Un modelo grande
en un equipo chico no se rompe: simplemente tarda muchísimo y el sistema se pone lento.

Abajo hay tres opciones. Deja activa una sola, la que corresponda a tu computadora, y comenta
las demás poniéndoles un signo de gato al inicio de la línea. Si no sabes cuánta memoria
tienes, quédate con la opción A, que funciona en cualquier equipo.

El modelo de embeddings no se elige por equipo: es ligero y va igual en todos. Sí conviene saber
de dónde salió esa elección, y la respuesta es que está medida con documentos en español; en la
práctica 3 vas a reproducir la medición y a ver a los tres candidatos compitiendo.

In [3]:
# ---- Opción A: equipos de 8 GB de memoria o menos (descarga 3.3 GB) ---------
MODELO_LLM = "gemma3:4b"

# ---- Opción B: equipos de 16 GB de memoria (descarga 10 GB) ----------------
# MODELO_LLM = "gemma4:12b"        # Windows y Linux
# MODELO_LLM = "gemma4:12b-mlx"    # Mac con chip Apple (M1 en adelante), va más rápido

# ---- Opción C: equipos de 32 GB de memoria o más (descarga 17 GB) ----------
# MODELO_LLM = "gemma4:26b"        # Windows y Linux
# MODELO_LLM = "gemma4:26b-mlx"    # Mac con chip Apple

# El modelo de embeddings es ligero y es el mismo para todos. La elección está
# medida, no copiada de un tutorial: lo comprobamos en la práctica 3. Se eligió
# éste porque es el único de los tres que encuentra un pasaje en inglés cuando la
# pregunta va en español, algo que hace falta en cuanto el corpus mezcla idiomas.
MODELO_EMBEDDINGS = "embeddinggemma:300m"

print(f"Modelo de lenguaje:   {MODELO_LLM}")
print(f"Modelo de embeddings: {MODELO_EMBEDDINGS}")
print("\nSi alguno no aparece en la lista de la celda anterior, descárgalo con:")
print(f"   ollama pull {MODELO_LLM}")
print(f"   ollama pull {MODELO_EMBEDDINGS}")

Modelo de lenguaje:   gemma3:4b
Modelo de embeddings: embeddinggemma:300m

Si alguno no aparece en la lista de la celda anterior, descárgalo con:
   ollama pull gemma3:4b
   ollama pull embeddinggemma:300m


## 4. El corpus de trabajo

Dos cambios respecto a la práctica anterior.

El primero es que sumamos un segundo documento, un catálogo interno con claves de producto,
códigos de estado de pedido y códigos de rechazo de pago. Lo necesitamos porque la búsqueda por
palabras se defiende justamente ahí, en los identificadores exactos, y sin ellos no habría nada
que comparar.

El segundo es que metemos un libro entero como ruido. Suena raro, pero tiene un motivo: con
veintitantos fragmentos, pedir tres es quedarse con medio corpus y todo parece funcionar. Con
varios cientos, elegir tres es una selección de verdad, que es la situación en la que un sistema
real trabaja.

In [4]:
import warnings
warnings.filterwarnings("ignore", message=".*langchain-community.*")

from pathlib import Path

import pymupdf
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


def tabla_a_frases(filas):
    encabezados = filas[0]
    return ["; ".join(f"{e}: {v}" for e, v in zip(encabezados, fila) if v)
            for fila in filas[1:]]


# Esta función repite en corto la preparación de la práctica 2: saca las tablas por un
# lado y el texto corrido por otro, y le pone a cada fragmento una etiqueta de origen.
def cargar(nombre, etiqueta):
    """Extrae un PDF separando tablas de texto corrido, como en la práctica 2."""
    doc = pymupdf.open(Path("documentos") / nombre)
    fragmentos = []
    divisor = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

    for pagina in doc:
        tablas = pagina.find_tables().tables
        for tabla in tablas:
            for frase in tabla_a_frases(tabla.extract()):
                fragmentos.append(Document(
                    page_content=frase, metadata={"fuente": etiqueta, "tipo": "tabla"}))

        recortes = [pagina.get_text(clip=t.bbox) for t in tablas]
        lineas = [l.strip() for l in pagina.get_text().split("\n")
                  if l.strip() and not any(l.strip() in r for r in recortes)]
        for trozo in divisor.split_text(" ".join(lineas)):
            fragmentos.append(Document(
                page_content=trozo, metadata={"fuente": etiqueta, "tipo": "texto"}))

    return fragmentos


# El corpus de trabajo: dos documentos reales de la tienda.
corpus = cargar("tiendasol_politicas.pdf", "politicas")
corpus += cargar("tiendasol_catalogo.pdf", "catalogo")
utiles = len(corpus)

# Y un libro que no tiene nada que ver, a propósito. Un corpus con solo dos documentos
# hace que cualquier búsqueda parezca buena: el relleno es lo que la pone a prueba.
ruido = cargar("Alice_in_Wonderland.pdf", "ruido")
corpus += ruido

print(f"Fragmentos de TiendaSol: {utiles}")
print(f"Fragmentos de relleno:   {len(ruido)}")
print(f"Corpus total:            {len(corpus)}")
print("\nEjemplos del catálogo:\n")
for f in corpus[:3]:
    if f.metadata["fuente"] == "catalogo":
        print(f"  {f.page_content[:80]}")
for f in corpus:
    if "TS-CAL" in f.page_content:
        print(f"  {f.page_content[:80]}")
        break

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


Fragmentos de TiendaSol: 27
Fragmentos de relleno:   450
Corpus total:            477

Ejemplos del catálogo:

  Clave: TS-CAL-4471; Articulo: Zapatilla urbana Andes; Linea: Calzado; Precio: S/


## 5. Los dos tipos de búsqueda

La que venimos usando se llama búsqueda densa, o por significado. Convierte la pregunta y los
fragmentos a vectores y compara direcciones. Su virtud es entender que "regresar un artículo" y
"devolver un producto" son lo mismo.

La otra se llama búsqueda léxica, o por palabras, y es la que usan los buscadores desde hace
décadas. Cuenta coincidencias de términos, pesando más los que son raros en el conjunto y menos
los comunes. La función más usada se llama BM25.

En teoría se reparten el trabajo así: la densa entiende intenciones pero se pierde con
identificadores; la léxica clava los identificadores pero no entiende sinónimos. Vamos a
comprobar si esa división de tareas es cierta.

In [5]:
import re
import unicodedata

from rank_bm25 import BM25Okapi


# La búsqueda léxica compara palabras, no significados. Antes hay que dejarlas todas
# en la misma forma: minúsculas, sin acentos y separadas una por una.
def tokenizar(texto):
    """Parte el texto en palabras, sin acentos y en minúsculas."""
    texto = unicodedata.normalize("NFKD", texto.lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.findall(r"[a-z0-9]+", texto)


# BM25 es el buscador de toda la vida, el de las palabras exactas. Pesa cada término
# según lo raro que sea en el corpus: un código como TS-CAL-4472 aparece en un solo
# fragmento, así que vale muchísimo cuando alguien lo escribe.
bm25 = BM25Okapi([tokenizar(d.page_content) for d in corpus])

print("Índice léxico construido.\n")
print("Así ve el buscador esta frase:\n")
ejemplo = "El código TS-CAL-4472 corresponde a la Zapatilla urbana Andes Pro"
print(f"  original: {ejemplo}")
print(f"  tokens:   {tokenizar(ejemplo)}")

Índice léxico construido.

Así ve el buscador esta frase:

  original: El código TS-CAL-4472 corresponde a la Zapatilla urbana Andes Pro
  tokens:   ['el', 'codigo', 'ts', 'cal', '4472', 'corresponde', 'a', 'la', 'zapatilla', 'urbana', 'andes', 'pro']


Fíjate en qué le pasó al código al tokenizarlo: `TS-CAL-4472` se partió en tres piezas.

Eso tiene una consecuencia que vas a ver más adelante. Si el cliente escribe el código sin
guiones, como `tscal4472`, la búsqueda por palabras no encuentra nada, porque para ella eso es
un término distinto que no aparece en ningún fragmento.

## 6. Las dos búsquedas, cara a cara

Ahora las definimos y las corremos sobre las mismas consultas.

In [6]:
from langchain_community.vectorstores import LanceDB
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=MODELO_EMBEDDINGS)
almacen = LanceDB.from_documents(corpus, embeddings, uri="lancedb_p4",
                                 table_name="hibrida", mode="overwrite")


# Búsqueda densa: la de los embeddings, la que entiende que devolver y regresar
# quieren decir lo mismo.
def buscar_densa(consulta, k=3):
    return [d.page_content for d in almacen.similarity_search(consulta, k=k)]


# Búsqueda léxica: la de las palabras exactas. No entiende sinónimos, pero tampoco
# se equivoca con un número de serie.
def buscar_lexica(consulta, k=3):
    puntajes = bm25.get_scores(tokenizar(consulta))
    mejores = sorted(range(len(puntajes)), key=lambda i: puntajes[i], reverse=True)[:k]
    return [corpus[i].page_content for i in mejores]


for consulta in ["¿Cuánto tiempo tengo para devolver algo?", "¿Qué significa EST-07?"]:
    print(f"CONSULTA: {consulta}")
    print(f"  densa  → {buscar_densa(consulta, 1)[0][:88]}")
    print(f"  léxica → {buscar_lexica(consulta, 1)[0][:88]}")
    print()

CONSULTA: ¿Cuánto tiempo tengo para devolver algo?
  densa  → la devolucion de un producto dentro de los treinta dias calendario siguientes a la entre
  léxica → confirma la recepcion del producto devuelto, el area de finanzas dispone de cinco dias h

CONSULTA: ¿Qué significa EST-07?


  densa  → Codigo: EST-07; Estado: En transito; Que significa: El paquete salio del almacen rumbo a
  léxica → Codigo: EST-07; Estado: En transito; Que significa: El paquete salio del almacen rumbo a



## 7. Combinar las dos

Si cada una es fuerte donde la otra es débil, lo sensato es usar las dos y juntar los resultados.
El problema es cómo: los puntajes no son comparables, porque una devuelve similitudes entre cero
y uno y la otra devuelve puntajes sin techo.

La solución más usada evita el problema por completo: en lugar de mezclar puntajes, mezcla
posiciones. Se llama fusión por rango recíproco, y es más simple de lo que suena. Cada fragmento
recibe puntos según el lugar en que quedó en cada lista, y los puntos valen menos conforme baja
la posición. Un fragmento que quedó primero en una lista y no aparece en la otra puede acabar por
encima de uno que quedó tercero en ambas.

La constante de 60 que verás es el valor habitual; sirve para que el primer lugar no aplaste al
resto.

In [7]:
# La híbrida no suma puntajes, porque los dos buscadores usan escalas distintas y no
# son comparables. Suma POSICIONES: un fragmento que quedó primero en cualquiera de
# las dos listas sube, y uno que quedó bien en las dos sube todavía más.
def buscar_hibrida(consulta, k=3, constante=60):
    """Fusión por rango recíproco de la búsqueda densa y la léxica."""
    candidatos_densa = buscar_densa(consulta, k * 2)
    candidatos_lexica = buscar_lexica(consulta, k * 2)

    # A cada texto se le da 1/(60 + posición). La constante 60 evita que el primer
    # lugar aplaste a todos los demás; es el valor habitual en la literatura.
    puntos = {}
    for lista in (candidatos_densa, candidatos_lexica):
        for posicion, texto in enumerate(lista, start=1):
            puntos[texto] = puntos.get(texto, 0) + 1 / (constante + posicion)

    ordenados = sorted(puntos.items(), key=lambda par: par[1], reverse=True)
    return [texto for texto, _ in ordenados[:k]]


consulta = "¿Qué producto es TS-CAL-4472?"
print(f"CONSULTA: {consulta}\n")
for i, texto in enumerate(buscar_hibrida(consulta), 1):
    print(f"  {i}. {texto[:92]}")

CONSULTA: ¿Qué producto es TS-CAL-4472?

  1. Clave: TS-CAL-4472; Articulo: Zapatilla urbana Andes Pro; Linea: Calzado; Precio: S/ 249.00
  2. Clave: TS-CAL-4471; Articulo: Zapatilla urbana Andes; Linea: Calzado; Precio: S/ 189.00
  3. TiendaSol · Catalogo y codigos de referencia Uso interno del area de atencion · version 3.0 


## 8. La pregunta que importa: ¿sirve de algo?

Ya está implementada. Ahora hay que averiguar si mejora algo, y para eso volvemos al método de
la práctica 3: un conjunto de preguntas cuyo fragmento correcto conocemos de antemano.

Las agrupamos en dos familias. Las semánticas, donde el cliente describe lo que quiere sin usar
las palabras del documento, y las de identificador, donde menciona un código exacto.

In [8]:
# Ocho consultas de dos familias distintas. Las semánticas se preguntan con palabras
# propias; las de código traen un identificador exacto. La comparación solo tiene
# sentido si se mira cada familia por separado.
PRUEBAS = [
    ("semántica", "¿Cuántos días tengo para devolver un producto?", "treinta dias"),
    ("semántica", "¿Quién paga el envío de una devolución?", "gastos de envio"),
    ("semántica", "¿Cuándo me devuelven el dinero?", "cinco dias habiles"),
    ("semántica", "¿Cuánto cuesta mandar algo a la selva?", "selva"),
    ("código", "¿Qué producto es TS-CAL-4472?", "ts-cal-4472"),
    ("código", "¿Qué significa EST-07?", "est-07"),
    ("código", "Mi pago falló con PAG-11", "pag-11"),
    ("código", "precio de TS-TER-9014", "ts-ter-9014"),
]


def sin_acentos(texto):
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in texto if not unicodedata.combining(c))


# Un acierto es que la palabra clave aparezca en los fragmentos recuperados. Se cuenta
# por familia para ver en qué es fuerte cada método, no solo quién gana en total.
def evaluar(funcion, k=3):
    marcador = {}
    for familia, consulta, clave in PRUEBAS:
        encontrado = sin_acentos(clave) in sin_acentos(" ".join(funcion(consulta, k)))
        marcador.setdefault(familia, [0, 0])
        marcador[familia][0] += encontrado
        marcador[familia][1] += 1
    return marcador


print(f"{'método':<12} {'semánticas':>12} {'códigos':>10} {'total':>9}")
print("-" * 46)
for nombre, funcion in [("densa", buscar_densa), ("léxica", buscar_lexica),
                        ("híbrida", buscar_hibrida)]:
    m = evaluar(funcion)
    total = sum(v[0] for v in m.values())
    n = sum(v[1] for v in m.values())
    print(f"{nombre:<12} {m['semántica'][0]}/{m['semántica'][1]:<10} "
          f"{m['código'][0]}/{m['código'][1]:<8} {total}/{n:<7}")

método         semánticas    códigos     total
----------------------------------------------


densa        4/4          4/4        8/8      
léxica       3/4          4/4        7/8      


híbrida      4/4          4/4        8/8      


Y aquí viene el resultado incómodo, que es el motivo por el que esta práctica existe.

La búsqueda híbrida no mejora nada. La densa, sola, ya acierta todo lo que hay que acertar,
incluidos los códigos que supuestamente eran su punto débil. La léxica, sola, es la que se queda
atrás.

Antes de encogerse de hombros conviene entender por qué, porque el motivo dice algo sobre cómo
envejecen los consejos técnicos.

La idea de que la búsqueda por significado se pierde con los identificadores viene de los
primeros modelos de embeddings, que efectivamente los diluían. Los modelos actuales conservan
bastante bien los términos raros y literales, así que `TS-CAL-4472` no se les escapa. El consejo
sigue circulando, pero el problema que lo motivaba se encogió.

Súmale que nuestro corpus tiene unos pocos cientos de fragmentos y que los códigos aparecen tal
cual en el texto. Es un escenario cómodo.

## 9. Entonces, ¿cuándo vale la pena?

No es que la búsqueda híbrida sea inútil; es que no hace falta en nuestro caso. Hay situaciones
donde sí cambia el resultado, y conviene reconocerlas:

Cuando el corpus es enorme. Con millones de fragmentos, el vecindario de cualquier consulta se
llena de textos parecidos y la coincidencia exacta de un término raro vuelve a ser una señal
valiosa.

Cuando el vocabulario es muy técnico. Nombres de medicamentos, referencias legales, números de
parte industriales. Ahí el término exacto es la pregunta entera.

Cuando necesitas explicar por qué salió un resultado. La búsqueda por palabras te deja señalar
la coincidencia; la densa solo puede decir que los vectores estaban cerca. En un área regulada,
esa diferencia puede ser lo que decide.

Y cuando el modelo de embeddings es débil o no cubre bien tu idioma, la búsqueda léxica funciona
de red de seguridad.

La forma de saberlo es la que acabas de usar: implementarla, medirla contra tu propio conjunto de
preguntas, y quedártela solo si gana. Una pieza más en el sistema es una pieza más que mantener,
que hace más lenta cada consulta y que puede fallar.

## 10. Un caso donde la léxica sí se rompe

Para que quede clara la asimetría, vale la pena ver el reverso: consultas escritas como las
escribe un cliente de verdad, con el código incompleto o mal tecleado.

In [9]:
# La prueba honesta: nadie escribe TS-CAL-4472 con los guiones en su lugar. Así es
# como llegan las consultas de verdad, y aquí es donde la búsqueda léxica se cae.
COMO_ESCRIBE_LA_GENTE = [
    ("mi codigo es TS CAL 4472", "ts-cal-4472"),
    ("tscal4472", "ts-cal-4472"),
    ("me sale el error pag 11", "pag-11"),
    ("la zapatilla Andes Pro cuanto cuesta", "ts-cal-4472"),
]

print(f"{'como lo escribe el cliente':<38} {'densa':>7} {'léxica':>8} {'híbrida':>9}")
print("-" * 64)
for consulta, clave in COMO_ESCRIBE_LA_GENTE:
    fila = []
    for funcion in (buscar_densa, buscar_lexica, buscar_hibrida):
        ok = sin_acentos(clave) in sin_acentos(" ".join(funcion(consulta, 3)))
        fila.append("sí" if ok else "no")
    print(f"{consulta[:36]:<38} {fila[0]:>7} {fila[1]:>8} {fila[2]:>9}")

como lo escribe el cliente               densa   léxica   híbrida
----------------------------------------------------------------


mi codigo es TS CAL 4472                    sí       sí        sí


tscal4472                                   sí       no        sí


me sale el error pag 11                     sí       sí        sí


la zapatilla Andes Pro cuanto cuesta        sí       sí        sí


Ahí está lo que anticipamos en la sección 5. Cuando el cliente escribe el código pegado, sin
guiones ni espacios, la búsqueda por palabras no encuentra nada: para ella `tscal4472` es un
término que no existe en ningún documento. La densa, en cambio, se las arregla.

Es un buen recordatorio de que la búsqueda léxica es literal en el peor sentido de la palabra.
Cualquier variante que no anticipaste, un guión de más, un acento, un plural, es un término
distinto.

## 11. La segunda etapa: reordenar

Cambiamos de técnica, y esta sí va a mostrar algo.

Los sistemas serios no buscan una vez y ya. Buscan de forma amplia y barata, trayendo bastantes
candidatos, y después reordenan ese puñado con un criterio más caro que no sería viable aplicar a
todo el corpus. Se llama recuperación en dos etapas.

Hay dos motivos para reordenar. El primero es la relevancia: acomodar los mejores arriba. El
segundo es menos obvio y suele importar más: la variedad.

In [10]:
import numpy as np


# Recuperar tres fragmentos que dicen lo mismo es peor que recuperar uno: se gasta el
# contexto del modelo en repetir. Esta función mide qué tan parecidos son entre sí.
def redundancia(textos):
    """Qué tan parecidos son entre sí los fragmentos recuperados.

    Cerca de 1 significa que dicen casi lo mismo y estamos gastando el contexto
    del modelo en repetir información.
    """
    vectores = np.array(embeddings.embed_documents(textos))
    vectores = vectores / np.linalg.norm(vectores, axis=1, keepdims=True)
    similitudes = vectores @ vectores.T
    n = len(textos)
    return (similitudes.sum() - n) / (n * (n - 1))


consulta = "¿Cuánto cuesta el envío?"
recuperados = buscar_densa(consulta, 4)

print(f"CONSULTA: {consulta}\n")
for i, texto in enumerate(recuperados, 1):
    print(f"  {i}. {texto[:86]}")
print(f"\nRedundancia entre ellos: {redundancia(recuperados):.3f}")

CONSULTA: ¿Cuánto cuesta el envío?

  1. Zona: Provincias, capital; Plazo habil: 3 a 5 dias; Costo: S/ 15.90; Envio gratis desd
  2. Zona: Zona de selva; Plazo habil: 8 a 12 dias; Costo: S/ 29.90; Envio gratis desde: S/
  3. Zona: Provincias, interior; Plazo habil: 5 a 8 dias; Costo: S/ 22.90; Envio gratis des
  4. Zona: Lima metropolitana; Plazo habil: 1 a 2 dias; Costo: S/ 9.90; Envio gratis desde:



Redundancia entre ellos: 0.781


Mira los cuatro fragmentos y verás el problema: dicen casi lo mismo. La búsqueda hizo su trabajo,
trajo los cuatro más parecidos a la pregunta, pero como se parecen a la pregunta también se
parecen entre sí.

El costo es real. Le estamos dando al modelo cuatro versiones del mismo dato en lugar de cuatro
datos, y desperdiciando tres cuartas partes del espacio disponible.

## 12. Reordenar por variedad

La técnica que corrige esto se llama relevancia marginal máxima. Elige los fragmentos de uno en
uno, y en cada paso no toma el más parecido a la pregunta, sino el que mejor equilibra dos cosas:
parecerse a la pregunta y no parecerse a lo que ya eligió.

Un parámetro controla la mezcla. Con `lambda_mult=1` se comporta como la búsqueda normal, y
mientras más baja, más peso le da a la variedad. LanceDB ya lo trae, así que no hay que
programarlo.

In [11]:
# Tres consultas amplias, del tipo que tiene respuesta repartida en varios fragmentos.
# Son las que más sufren la redundancia.
CONSULTAS = [
    "¿Cuánto cuesta el envío?",
    "¿Cómo hago una devolución?",
    "¿Qué códigos de error existen?",
]

print(f"{'consulta':<34} {'normal':>8} {'variada':>9} {'cambio':>9}")
print("-" * 62)
for consulta in CONSULTAS:
    normales = buscar_densa(consulta, 4)
    # MMR busca primero 20 candidatos (fetch_k) y de ahí elige 4 que sean a la vez
    # pertinentes y distintos entre sí. lambda_mult=0.5 reparte el peso mitad y mitad
    # entre parecerse a la pregunta y no parecerse a lo ya elegido.
    variados = [d.page_content for d in almacen.max_marginal_relevance_search(
        consulta, k=4, fetch_k=20, lambda_mult=0.5)]

    antes, despues = redundancia(normales), redundancia(variados)
    print(f"{consulta[:32]:<34} {antes:>8.3f} {despues:>9.3f} {despues - antes:>+9.3f}")

consulta                             normal   variada    cambio
--------------------------------------------------------------


¿Cuánto cuesta el envío?              0.781     0.158    -0.623


¿Cómo hago una devolución?            0.531     0.181    -0.350


¿Qué códigos de error existen?        0.739     0.164    -0.575


Esta vez la mejora no admite discusión: la redundancia se desploma. Los fragmentos que recibe el
modelo dejan de ser variaciones del mismo párrafo y pasan a cubrir aspectos distintos de la
pregunta.

Y nota la diferencia con el caso de la búsqueda híbrida, porque es la moraleja de la práctica. Las
dos técnicas venían igual de recomendadas. Al medirlas con nuestro corpus, una no aportó nada y la
otra mejoró de forma clara. No había manera de saber cuál era cuál sin probarlas.

Cuándo conviene bajar el parámetro de variedad: cuando resumes opiniones o reseñas y quieres
cubrir posturas distintas, o cuando el corpus tiene mucha información repetida entre documentos,
que es lo normal en bases de conocimiento que crecieron por acumulación. Cuándo conviene
mantenerlo alto: cuando la pregunta tiene una respuesta única y precisa, donde traer variedad es
traer ruido.

## 13. Reordenar con el modelo

La otra forma de reordenar es preguntarle al propio modelo de lenguaje cuáles de los candidatos
responden mejor. Es caro, porque cada consulta implica una llamada más, pero no requiere instalar
nada nuevo y captura matices que la similitud de vectores no ve.

En sistemas de producción esto se hace con modelos especializados, entrenados solo para puntuar
pares de pregunta y fragmento. Aquí lo hacemos con el modelo que ya tenemos, que es la alternativa
razonable cuando no quieres sumar otra pieza a la instalación.

In [12]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

# El modelo que redacta la respuesta. temperature=0 hace que, ante la misma pregunta,
# conteste siempre lo mismo: justo lo que se quiere en atención al cliente.
#
# Aquí es donde una organización pondría la llamada a un servicio comercial si
# decidiera no usar un modelo local. Este curso corre en local a propósito.
llm = ChatOllama(model=MODELO_LLM, temperature=0)

# El reordenamiento es una segunda pasada: la búsqueda trae candidatos rápido y
# barato, y el modelo los ordena despacio pero con criterio. Se le piden solo números
# porque son fáciles de interpretar sin adivinar.
ORDENADOR = ChatPromptTemplate.from_template(
    """Tienes una pregunta de un cliente y varios fragmentos numerados de la
documentación. Ordena los fragmentos del más útil al menos útil para responder
esa pregunta.

Responde solo con los números separados por comas, por ejemplo: 3,1,4,2

Pregunta: {pregunta}

Fragmentos:
{fragmentos}"""
)


# Nunca hay que confiar en que el modelo devuelva el formato pedido. Aquí se sacan los
# números que haya, se descartan los inválidos y los repetidos, y lo que el modelo
# haya olvidado se agrega al final en su orden original.
def reordenar_con_modelo(consulta, candidatos):
    numerados = "\n".join(f"[{i}] {t[:200]}" for i, t in enumerate(candidatos, 1))
    respuesta = (ORDENADOR | llm | StrOutputParser()).invoke(
        {"pregunta": consulta, "fragmentos": numerados})

    orden = []
    for pieza in re.findall(r"\d+", respuesta):
        indice = int(pieza) - 1
        if 0 <= indice < len(candidatos) and indice not in orden:
            orden.append(indice)
    # Cualquier candidato que el modelo haya olvidado va al final, en su orden original
    orden += [i for i in range(len(candidatos)) if i not in orden]
    return [candidatos[i] for i in orden]


consulta = "¿Cuánto tiempo tarda en llegarme el reembolso?"
candidatos = buscar_densa(consulta, 5)
reordenados = reordenar_con_modelo(consulta, candidatos)

print(f"CONSULTA: {consulta}\n")
print("Orden de la búsqueda:")
for i, t in enumerate(candidatos, 1):
    print(f"  {i}. {t[:78]}")
print("\nOrden después del reordenamiento:")
for i, t in enumerate(reordenados, 1):
    print(f"  {i}. {t[:78]}")

CONSULTA: ¿Cuánto tiempo tarda en llegarme el reembolso?

Orden de la búsqueda:
  1. confirma la recepcion del producto devuelto, el area de finanzas dispone de ci
  2. la devolucion de un producto dentro de los treinta dias calendario siguientes 
  3. de entrega se cuentan en dias habiles a partir de la confirmacion del pago, no
  4. Codigo: EST-12; Estado: Devolucion en curso; Que significa: El cliente inicio 
  5. y elige la opcion Solicitar devolucion. El sistema genera una guia de retorno 

Orden después del reordenamiento:
  1. confirma la recepcion del producto devuelto, el area de finanzas dispone de ci
  2. Codigo: EST-12; Estado: Devolucion en curso; Que significa: El cliente inicio 
  3. la devolucion de un producto dentro de los treinta dias calendario siguientes 
  4. de entrega se cuentan en dias habiles a partir de la confirmacion del pago, no
  5. y elige la opcion Solicitar devolucion. El sistema genera una guia de retorno 


Compara las dos listas. Si el modelo movió hacia arriba el fragmento que de verdad contesta, el
reordenamiento sirvió. Si dejó todo igual, quiere decir que la búsqueda ya venía bien ordenada,
que es lo que suele pasar con corpus pequeños.

Antes de adoptarlo, dos costos que conviene tener presentes. Cada consulta necesita una llamada
extra al modelo, así que la respuesta tarda más, y en un chat de atención eso se siente. Y el
modelo puede desordenar cosas que estaban bien, porque no es un ordenador confiable sino un
generador de texto al que le pedimos números.

Por eso los sistemas serios usan modelos especializados en esta tarea, que son más chicos, más
rápidos y más consistentes que un modelo de propósito general.

## 14. Lo que aprendiste hoy, además de dos técnicas

Implementaste búsqueda por palabras, fusión de resultados, reordenamiento por variedad y
reordenamiento con el modelo. Son las piezas que separan un prototipo de un sistema serio.

Pero lo que de verdad conviene llevarse es lo que pasó al medirlas.

Las dos técnicas llegaron con la misma recomendación. La búsqueda híbrida, que es la más citada
de las dos, no aportó nada en nuestro caso, porque el problema que resuelve se encogió con los
modelos actuales y porque nuestro corpus es pequeño y sus códigos aparecen tal cual. El
reordenamiento por variedad, en cambio, mejoró de forma clara y medible.

Ninguna cantidad de lectura te habría dicho cuál era cuál. Solo la medición.

Ese es el hábito que vale: cada pieza que agregas al sistema hay que justificarla con una
medición sobre tus propios documentos y tus propias preguntas. Lo que no gana, no entra, porque
todo lo que entra hay que mantenerlo, hace más lenta cada consulta y puede fallar.

En la práctica siguiente cambiamos de tema. Ya tenemos un sistema que encuentra bien; toca
ocuparse de que no diga cosas que no debe.